# 02 — Customer Segmentation
## Clustering the Telco Customer Base

**Objective:** Group customers into meaningful segments to enable targeted retention strategies.

## 1. Setup & Data Preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

import sys
sys.path.append('..')
from src.preprocessing import load_data, create_service_features, create_demographic_features
from src.segmentation import prepare_segmentation_data, find_optimal_k, run_kmeans, apply_pca, label_segments, SEGMENT_NAMES

In [ ]:
df = load_data()
df = create_demographic_features(df)
df = create_service_features(df)
print(f"Loaded {len(df)} customers")

## 2. Optimal Number of Segments

In [ ]:
seg_data, scaler = prepare_segmentation_data(df)
inertias = find_optimal_k(seg_data, max_k=10)

plt.figure(figsize=(10, 5))
plt.plot(range(1, 11), inertias, 'bo-', linewidth=2)
plt.axvline(x=5, color='red', linestyle='--', alpha=0.5, label='k=5 (selected)')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.legend()
plt.savefig('../reports/elbow_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. K-Means Clustering

In [ ]:
labels, kmeans = run_kmeans(seg_data, n_clusters=5)
df['Cluster'] = labels
df['Segment'] = label_segments(labels)
print("Cluster distribution:")
print(df['Segment'].value_counts())

In [ ]:
pca_components, pca = apply_pca(seg_data)
df['PCA1'] = pca_components[:, 0]
df['PCA2'] = pca_components[:, 1]

plt.figure(figsize=(12, 8))
palette = {0: '#e74c3c', 1: '#2ecc71', 2: '#3498db', 3: '#f39c12', 4: '#9b59b6'}
for cluster in sorted(df['Cluster'].unique()):
    subset = df[df['Cluster'] == cluster]
    plt.scatter(subset['PCA1'], subset['PCA2'],
                c=palette[cluster], label=SEGMENT_NAMES.get(cluster, f'Segment {cluster}'),
                alpha=0.6, s=20)
plt.xlabel(f'PCA Component 1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PCA Component 2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title('Customer Segments Visualized (PCA)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../reports/segmentation_pca.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Segment Profiles

In [ ]:
segment_summary = df.groupby('Segment').agg(
    Count=('tenure', 'size'),
    AvgTenure=('tenure', 'mean'),
    AvgMonthlyCharges=('MonthlyCharges', 'mean'),
    AvgTotalCharges=('TotalCharges', 'mean'),
    AvgServiceCount=('ServiceCount', 'mean'),
    PctWithDependents=('HasDependents', 'mean'),
    PctSenior=('SeniorCitizen', 'mean'),
).round(2)
segment_summary['ChurnRate'] = df.groupby('Segment')['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100).round(1)
segment_summary['PctOfBase'] = (segment_summary['Count'] / len(df) * 100).round(1)
segment_summary

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
metrics = ['AvgTenure', 'AvgMonthlyCharges', 'AvgServiceCount', 'ChurnRate', 'Count']
for i, metric in enumerate(metrics):
    ax = axes[i // 3, i % 3]
    segment_summary[metric].sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'Segment Comparison: {metric}')
    ax.set_xlabel(metric)

axes[1, 2].axis('off')
plt.tight_layout()
plt.savefig('../reports/segment_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Segment Deep Dive

### Segment Descriptions

In [ ]:
segment_descriptions = df.groupby('Segment').agg({
    'tenure': ['mean', 'median', 'min', 'max'],
    'MonthlyCharges': ['mean', 'std'],
    'Churn': lambda x: (x == 'Yes').mean() * 100,
    'SeniorCitizen': 'mean',
    'Partner': lambda x: (x == 'Yes').mean(),
    'Dependents': lambda x: (x == 'Yes').mean(),
}).round(2)
segment_descriptions.columns = ['AvgTenure', 'MedTenure', 'MinTenure', 'MaxTenure',
                                 'AvgMonthlyCharges', 'StdCharges',
                                 'ChurnRate%', 'SeniorCitizen%', 'HasPartner%', 'HasDependents%']
segment_descriptions

### Segment Interpretations

1. **High-Value Loyal** — Long tenure, high spend, low churn. These are your best customers. Protect them.
2. **Price-Sensitive** — Lower spend, moderate tenure. May respond to discounts.
3. **New / Short-Tenure** — Low tenure, varying spend. High churn risk. Need early engagement.
4. **Low Engagement** — Few services, lower spend. May not be fully adopted.
5. **Premium Power Users** — High tenure, high spend, many services. Low churn but high potential loss if they leave.

In [ ]:
print("Key Takeaways:")
print("=" * 50)
for segment in segment_summary.index:
    churn = segment_summary.loc[segment, 'ChurnRate']
    pct = segment_summary.loc[segment, 'PctOfBase']
    revenue_impact = segment_summary.loc[segment, 'Count'] * churn / 100 * segment_summary.loc[segment, 'AvgMonthlyCharges']
    print(f"{segment}: {pct:.1f}% of base, {churn:.1f}% churn, ~${revenue_impact:.0f}/mo at risk")

---
*End of 02 — Customer Segmentation*